# Can fine-tuned Laya beat Phi-4-mini on BoolQ?

Follow-up to [`2026-09-laya-vs-phi4-boolq`](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-laya-vs-phi4-boolq). On the full BoolQ validation split (3,270 examples), zero-shot Laya (421M) scored 75.7% and zero-shot Phi-4-mini-instruct (3.8B) scored 81.4%.

**Question:** if Laya is fine-tuned on 3,000 BoolQ training examples, does it close the gap with, or beat, an off-the-shelf Phi-4-mini?

**Setup**
- **Train:** 3,000 examples sampled from BoolQ's `train` split (seed 42). Another 300 disjoint train examples are held out for calibration and threshold selection.
- **Test:** the **full `validation` split (3,270)**, the same set as the previous experiment. No validation example is ever used for training, calibration or tuning.
- **Laya fine-tuning:** Laya's own RLCD recipe, adapted from [the official fine-tuning notebook](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb) to a single T4. It combines a policy-gradient term with a proper-scoring-rule reward and a soft cross-entropy term, followed by temperature calibration on the held-out slice.
- **Phi-4-mini:** zero-shot and **not** fine-tuned, with the same prompt and P(yes) extraction as before. Its predictions are reused from the previous run's `results.csv` (committed here as `prior_results.csv`), and only examples missing from it are run.
- All three models (base Laya, fine-tuned Laya and Phi-4-mini) see identical passage text (passages over 400 Laya tokens are truncated for all of them).

**Runtime → Change runtime type → T4 GPU**, then **Run all**. Expect about 10 min of training, about 5 min of Laya evaluation, and nothing for Phi-4-mini, since its predictions are reused. Everything is cached, so a rerun after a disconnect resumes where it stopped.

In [ ]:
%pip install -q "laya>=0.3.20" "transformers==4.57.3" "accelerate==1.11.0" "datasets==4.8.5" safetensors scikit-learn scipy

In [ ]:
import gc, json, math, os, random, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch, transformers, laya
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "Fine-tuning needs a GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

## 1. Config

In [ ]:
N_TRAIN = 3000                # BoolQ train examples used for fine-tuning
N_CALIB = 300                 # disjoint train examples for temperature fitting + threshold choice
PASSAGE_TOKEN_BUDGET = 400    # same truncation as the previous experiment, applied to every model
THRESHOLD = 0.5

# Laya fine-tuning (official recipe, single GPU: effective batch = MICRO_BATCH * GRAD_ACCUM = 64)
LAYA_ID = "convaiinnovations/laya"
EPOCHS = 3
MICRO_BATCH = 8
GRAD_ACCUM = 8
GROUP_SIZE = 4                # noisy samples per item for the policy-gradient baseline
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1

# Phi-4-mini (zero-shot, identical to the previous experiment)
PHI_ID = "microsoft/Phi-4-mini-instruct"
PHI_BATCH = 8
SYSTEM_PROMPT = "You answer yes/no questions about a passage."
USER_TEMPLATE = "Passage:\n{passage}\n\nQuestion: {question}\nAnswer with only Yes or No."
# results.csv from the previous notebook (full validation run), committed next to this notebook.
# Fetched from GitHub when missing locally (e.g. on Colab); if that fails, Phi-4-mini is run here.
PRIOR_RESULTS = Path("prior_results.csv")
PRIOR_RESULTS_URL = ("https://raw.githubusercontent.com/CJosh88/slms/main/"
                     "experiments/2026-09-laya-finetune-boolq/prior_results.csv")

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/slms/2026-09-laya-finetune-boolq")
else:
    BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "cache"
FIG_DIR = BASE_DIR / "figures"
FT_DIR = BASE_DIR / "laya_boolq_ft"     # fine-tuned checkpoint (~850 MB, not committed)
for d in (CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Data

In [ ]:
def normalise_question(q: str) -> str:
    # BoolQ questions are lowercase with no question mark.
    q = q.strip()
    return q[0].upper() + q[1:] + ("" if q.endswith("?") else "?")


def to_frame(ds, idx):
    return pd.DataFrame({
        "idx": idx,
        "question": [normalise_question(ds[i]["question"]) for i in idx],
        "passage": [ds[i]["passage"] for i in idx],
        "label": [bool(ds[i]["answer"]) for i in idx],
    })


train_raw = load_dataset("google/boolq", split="train")
val_raw = load_dataset("google/boolq", split="validation")

order = np.random.default_rng(SEED).permutation(len(train_raw)).tolist()
train_df = to_frame(train_raw, sorted(order[:N_TRAIN]))
calib_df = to_frame(train_raw, sorted(order[N_TRAIN:N_TRAIN + N_CALIB]))
test_df = to_frame(val_raw, list(range(len(val_raw))))
assert not set(train_df.idx) & set(calib_df.idx)

MAJORITY = test_df.label.mean()
for name, d in [("train", train_df), ("calib", calib_df), ("test (validation)", test_df)]:
    print(f"{name:18s} {len(d):5d} examples | Yes rate {d.label.mean():.1%}")

## 3. Load base Laya and tokenise

The tokenizer, config and `build_sequence` come from the `laya` package, so training sequences are built exactly as the library builds them at inference.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

model_dir = snapshot_download(LAYA_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    base_cfg = json.load(f)
print({k: base_cfg[k] for k in ("max_len", "head_max_len") if k in base_cfg}, "| QTYPES", QTYPES)
print("base temperature:", base_cfg.get("temperature"))


def truncate(text, budget=PASSAGE_TOKEN_BUDGET):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= budget else tok.decode(ids[:budget])


for d in (train_df, calib_df, test_df):
    d["passage_in"] = [truncate(p) for p in d.passage]
    d["truncated"] = d.passage_in != d.passage
print(f"Truncated passages: train {train_df.truncated.mean():.1%}, test {test_df.truncated.mean():.1%}")

In [ ]:
NOUL = {"t": "noul", "crit": {}}
K_NOUL = len(render_options(NOUL))


def build_item(passage, question, label):
    # Target order for noul is [false, true], as in the official recipe.
    seq, markers = build_sequence(tok, passage, {**NOUL, "ins": question}, base_cfg["max_len"], base_cfg["head_max_len"])
    if len(markers) != K_NOUL:
        return None
    target = [0.0, 1.0] if label else [1.0, 0.0]
    return {"ids": seq, "markers": markers, "qtype": QTYPES["noul"], "target": target, "label": int(label)}


def build_items(d):
    items = [build_item(p, q, y) for p, q, y in zip(d.passage_in, d.question, d.label)]
    kept = [it for it in items if it is not None]
    print(f"  built {len(kept)} / {len(items)} items, max length {max(len(it['ids']) for it in kept)} tokens")
    return kept


print("train:"); train_items = build_items(train_df)
print("calib:"); calib_items = build_items(calib_df)

## 4. Baseline: zero-shot Laya on the full validation split

This re-scores base Laya under exactly the conditions used for the fine-tuned model, which makes the before/after comparison clean.

In [ ]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["idx"]: rec for rec in map(json.loads, f)}


def run_laya(agent, d, cache_path, desc):
    done = load_cache(cache_path)
    todo = d[~d.idx.isin(done)]
    print(f"{desc}: {len(done)} cached, {len(todo)} to run")
    torch.cuda.reset_peak_memory_stats()
    with cache_path.open("a", encoding="utf-8") as f:
        for _, r in tqdm(todo.iterrows(), total=len(todo), desc=desc):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            p = agent.predict(r.passage_in, {"q": {"type": "noul", "instructions": r.question}})["answers"]["q"]["noul"]
            torch.cuda.synchronize()
            f.write(json.dumps({"idx": int(r.idx), "p_yes": float(p), "latency_s": time.perf_counter() - t0}) + "\n")
    return torch.cuda.max_memory_allocated() / 1e9


base_agent = laya.load(LAYA_ID, device=DEVICE)
mem_base = run_laya(base_agent, test_df, CACHE_DIR / "laya_base.jsonl", "Laya base")
# P(yes) on the calibration slice too, for a like-for-like threshold choice later
run_laya(base_agent, calib_df, CACHE_DIR / "laya_base_calib.jsonl", "Laya base (calib)")
base_acc = np.mean([(r["p_yes"] >= THRESHOLD) == y for r, y in
                    zip(map(load_cache(CACHE_DIR / "laya_base.jsonl").get, test_df.idx), test_df.label)])
print(f"Zero-shot Laya validation accuracy: {base_acc:.1%}")
del base_agent; gc.collect(); torch.cuda.empty_cache()

## 5. Fine-tune Laya

A single-GPU port of the official RLCD training loop. Each step samples `GROUP_SIZE` noisy versions of the logits and scores them with a proper scoring rule. The policy-gradient loss pushes the logits toward better-scoring noise, and a soft cross-entropy term pulls them toward the label. Exploration noise anneals from 0.4 to 0.1. Held-out calibration accuracy is printed after every epoch.

Skipped if a finished checkpoint already exists in `FT_DIR`.

In [ ]:
from safetensors.torch import load_file, save_file
from laya.common import build_model, proper_reward


def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :k] = torch.tensor(it["target"])
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items])}


def forward(model, batch):
    with torch.autocast("cuda", dtype=torch.float16):
        logits, act = model(*(batch[k].to(DEVICE) for k in
                              ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")))
    return logits.float(), act


@torch.no_grad()
def raw_logits(model, items, bs=16):
    model.eval()
    out = []
    for i in range(0, len(items), bs):
        logits, _ = forward(model, collate(items[i:i + bs], tok.pad_token_id))
        out.append(logits[:, :K_NOUL].cpu())
    model.train()
    return torch.cat(out)


def fit_temperature(z, target):
    # Single temperature minimising cross-entropy on held-out items (as in the official recipe).
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(target * torch.log_softmax(z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0))


ft_done = (FT_DIR / "rl_agent_config.json").exists()
history = json.loads((CACHE_DIR / "train_history.json").read_text()) if ft_done else []
if not ft_done:
    cfg = dict(base_cfg)
    cfg["gradient_checkpointing"] = True
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(DEVICE).train()

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                                   {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    total_updates = math.ceil(len(train_items) / (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    calib_y = torch.tensor([it["label"] for it in calib_items])
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    for epoch in range(EPOCHS):
        rng = random.Random(SEED + epoch)
        items = train_items[:]
        rng.shuffle(items)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * epoch / max(1, EPOCHS - 1)
        optimizer.zero_grad(set_to_none=True)
        losses, rewards = [], []
        n_micro = math.ceil(len(items) / MICRO_BATCH)
        for step, b in enumerate(tqdm(range(0, len(items), MICRO_BATCH), desc=f"epoch {epoch + 1}/{EPOCHS}")):
            batch = collate(items[b:b + MICRO_BATCH], tok.pad_token_id)
            logits, act = forward(model, batch)
            mask = batch["marker_mask"].to(DEVICE)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(DEVICE)
            qtype = batch["qtype"].to(DEVICE)

            # Zero-mean Gaussian exploration on the logits, scored with a proper scoring rule
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=DEVICE) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask, w_sph=0.75, w_rps=1.0)
                adv = (r - r.mean(0, keepdim=True)) / ((r - r.mean(0, keepdim=True)).std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == n_micro:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            losses.append(loss_ce.item()); rewards.append(r.mean().item())

        zc = raw_logits(model, calib_items)
        calib_acc = (zc.argmax(-1) == calib_y).float().mean().item()
        history.append({"epoch": epoch + 1, "ce_loss": float(np.mean(losses)), "reward": float(np.mean(rewards)),
                        "calib_acc": calib_acc, "minutes": (time.time() - t_start) / 60})
        print(history[-1])

    train_minutes = (time.time() - t_start) / 60
    train_peak_gb = torch.cuda.max_memory_allocated() / 1e9

    # Temperature calibration on the held-out slice, then save in the layout laya.Agent loads
    zc = raw_logits(model, calib_items)
    t_noul = fit_temperature(zc, torch.tensor([it["target"] for it in calib_items]))
    temps = base_cfg.get("temperature", [1.0, 1.0, 1.0])
    temps = list(temps) if isinstance(temps, (list, tuple)) else [float(temps)] * 3
    temps[QTYPES["noul"]] = t_noul
    print(f"Fitted noul temperature: {t_noul:.3f}")

    FT_DIR.mkdir(parents=True, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
              str(FT_DIR / "model.safetensors"))
    model.encoder.config.save_pretrained(FT_DIR / "encoder")
    tok.save_pretrained(FT_DIR / "tokenizer")
    save_cfg = {**base_cfg, "fine_tuned": True, "model_name": "laya-boolq", "temperature": temps}
    save_cfg.pop("temperature_by_options", None)
    (FT_DIR / "rl_agent_config.json").write_text(json.dumps(save_cfg, indent=2))
    (CACHE_DIR / "train_history.json").write_text(json.dumps(history))
    (CACHE_DIR / "train_meta.json").write_text(json.dumps(
        {"train_minutes": train_minutes, "train_peak_gb": train_peak_gb, "noul_temperature": t_noul,
         "n_train_items": len(train_items), "epochs": EPOCHS}))
    del model, optimizer, scheduler, scaler
    gc.collect(); torch.cuda.empty_cache()

train_meta = json.loads((CACHE_DIR / "train_meta.json").read_text())
print(train_meta)
pd.DataFrame(history)

## 6. Evaluate fine-tuned Laya on the full validation split

In [ ]:
ft_agent = laya.Agent(str(FT_DIR), device=DEVICE)
mem_ft = run_laya(ft_agent, test_df, CACHE_DIR / "laya_ft.jsonl", "Laya fine-tuned")
run_laya(ft_agent, calib_df, CACHE_DIR / "laya_ft_calib.jsonl", "Laya fine-tuned (calib)")
del ft_agent; gc.collect(); torch.cuda.empty_cache()

## 7. Phi-4-mini (zero-shot, not fine-tuned)

The code and prompt are identical to the previous experiment. Predictions from its `results.csv` (`PRIOR_RESULTS`) are imported into the cache first, and only examples still missing are generated. Phi-4-mini is also run on the calibration slice so that its threshold can be tuned the same way as Laya's.

In [ ]:
from transformers import AutoModelForCausalLM


def first_token_ids(tk, words):
    return sorted({tk.encode(w, add_special_tokens=False)[0] for w in words})


def parse_answer(text):
    t = text.strip().lower().lstrip("*'\"").strip()
    return True if t.startswith("yes") else False if t.startswith("no") else None


phi_test_path, phi_calib_path = CACHE_DIR / "phi4.jsonl", CACHE_DIR / "phi4_calib.jsonl"
if not PRIOR_RESULTS.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(PRIOR_RESULTS_URL, PRIOR_RESULTS)
    except Exception as e:
        print("Could not fetch prior results, Phi-4-mini will be run:", e)
if PRIOR_RESULTS.exists() and not phi_test_path.exists():
    prior = pd.read_csv(PRIOR_RESULTS)
    prior = prior[prior.model == "Phi-4-mini"]
    with phi_test_path.open("w", encoding="utf-8") as f:
        for r in prior.itertuples():
            f.write(json.dumps({"idx": int(r.idx), "output": str(r.output), "p_yes": float(r.p_yes),
                                "latency_s": float(r.latency_s)}) + "\n")
    print(f"Imported {len(prior)} Phi-4-mini predictions from {PRIOR_RESULTS}")

todo = {path: d[~d.idx.isin(load_cache(path))] for path, d in
        [(phi_test_path, test_df), (phi_calib_path, calib_df)]}
print({p.name: len(t) for p, t in todo.items()}, "examples to generate")

phi_meta_path = CACHE_DIR / "phi4_meta.json"
if any(len(t) for t in todo.values()):
    ptok = AutoTokenizer.from_pretrained(PHI_ID)
    ptok.padding_side = "left"
    if ptok.pad_token is None:
        ptok.pad_token = ptok.eos_token
    model = AutoModelForCausalLM.from_pretrained(PHI_ID, dtype=DTYPE, device_map="auto").eval()
    YES_IDS = first_token_ids(ptok, ["Yes", " Yes", "yes", " yes", "YES"])
    NO_IDS = first_token_ids(ptok, ["No", " No", "no", " no", "NO"])
    torch.cuda.reset_peak_memory_stats()
    for path, d in todo.items():
        with path.open("a", encoding="utf-8") as f:
            for start in tqdm(range(0, len(d), PHI_BATCH), desc=f"Phi-4-mini {path.stem}"):
                batch = d.iloc[start:start + PHI_BATCH]
                prompts = [ptok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": USER_TEMPLATE.format(passage=p, question=q)}],
                    tokenize=False, add_generation_prompt=True) for p, q in zip(batch.passage_in, batch.question)]
                enc = ptok(prompts, return_tensors="pt", padding=True).to(model.device)
                torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.no_grad():
                    out = model.generate(**enc, max_new_tokens=3, do_sample=False, output_scores=True,
                                         return_dict_in_generate=True, pad_token_id=ptok.pad_token_id)
                torch.cuda.synchronize(); per = (time.perf_counter() - t0) / len(batch)
                logits = out.scores[0].float()
                p_yes = torch.sigmoid(torch.logsumexp(logits[:, YES_IDS], -1) - torch.logsumexp(logits[:, NO_IDS], -1))
                texts = ptok.batch_decode(out.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
                for idx, t, p in zip(batch.idx, texts, p_yes.tolist()):
                    f.write(json.dumps({"idx": int(idx), "output": t, "p_yes": p, "latency_s": per}) + "\n")
    phi_meta_path.write_text(json.dumps({"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
                                         "params_m": sum(p.numel() for p in model.parameters()) / 1e6}))
    del model; gc.collect(); torch.cuda.empty_cache()
phi_meta = json.loads(phi_meta_path.read_text()) if phi_meta_path.exists() else {"peak_mem_gb": float("nan"), "params_m": 3836.0}

## 8. Results

In [ ]:
from sklearn.metrics import brier_score_loss, f1_score, roc_auc_score


def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


def ece(y, p, bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    which = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1)
    return sum(abs(y[which == b].mean() - p[which == b].mean()) * (which == b).mean()
               for b in range(bins) if (which == b).any())


def best_threshold(p, y):
    # Chosen on the calibration slice (train split), never on the test set.
    ths = np.round(np.arange(0.05, 0.96, 0.01), 2)
    return float(ths[np.argmax([((p >= t) == y).mean() for t in ths])])


MODELS = {
    "Laya (zero-shot)": ("laya_base", {"params_m": 421.0, "peak_mem_gb": mem_base}),
    "Laya (fine-tuned, 3k)": ("laya_ft", {"params_m": 421.0, "peak_mem_gb": mem_ft}),
    "Phi-4-mini (zero-shot)": ("phi4", phi_meta),
}
frames, tuned = [], {}
for name, (stem, meta) in MODELS.items():
    g = test_df[["idx", "label", "truncated"]].merge(pd.DataFrame(load_cache(CACHE_DIR / f"{stem}.jsonl").values()), on="idx")
    c = calib_df[["idx", "label"]].merge(pd.DataFrame(load_cache(CACHE_DIR / f"{stem}_calib.jsonl").values()), on="idx")
    tuned[name] = best_threshold(c.p_yes.values, c.label.values)
    if stem == "phi4":
        parsed = g.output.map(parse_answer)
        g["pred"] = [a if a is not None else p >= THRESHOLD for a, p in zip(parsed, g.p_yes)]
    else:
        g["pred"] = g.p_yes >= THRESHOLD
    g["pred_tuned"] = g.p_yes >= tuned[name]
    g["model"] = name
    frames.append(g)
res = pd.concat(frames, ignore_index=True)
res["correct"] = res.pred == res.label
res["correct_tuned"] = res.pred_tuned == res.label
assert res.groupby("model").size().eq(len(test_df)).all(), res.groupby("model").size()
res.drop(columns=[c for c in ["output"] if c in res]).to_csv(BASE_DIR / "results.csv", index=False)

rows = [{"model": "Majority class", "n": len(test_df), "accuracy": max(MAJORITY, 1 - MAJORITY)}]
for name, g in res.groupby("model", sort=False):
    lo, hi = wilson(g.correct.sum(), len(g))
    meta = MODELS[name][1]
    rows.append({
        "model": name, "n": len(g), "accuracy": g.correct.mean(), "ci_low": lo, "ci_high": hi,
        "acc_tuned_threshold": g.correct_tuned.mean(), "threshold_tuned": tuned[name],
        "f1_yes": f1_score(g.label, g.pred), "roc_auc": roc_auc_score(g.label, g.p_yes),
        "brier": brier_score_loss(g.label, g.p_yes), "ece": ece(g.label, g.p_yes),
        "ms_per_example": 1000 * g.latency_s.mean(), "params_m": meta["params_m"], "peak_mem_gb": meta["peak_mem_gb"],
    })
summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(BASE_DIR / "summary.csv")
print(f"Fine-tuning: {train_meta['train_minutes']:.1f} min, peak {train_meta['train_peak_gb']:.1f} GB, "
      f"{train_meta['n_train_items']} examples x {train_meta['epochs']} epochs")
summary.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "acc_tuned_threshold": "{:.1%}",
    "threshold_tuned": "{:.2f}", "f1_yes": "{:.3f}", "roc_auc": "{:.3f}", "brier": "{:.3f}", "ece": "{:.3f}",
    "ms_per_example": "{:.1f}", "params_m": "{:,.0f}", "peak_mem_gb": "{:.2f}",
}, na_rep="–")

### Verdict: fine-tuned Laya vs Phi-4-mini

The comparison is paired, since both models answered all 3,270 questions. The exact McNemar test runs on the examples where exactly one model was right.

In [ ]:
from scipy.stats import binomtest

wide = res.pivot(index="idx", columns="model", values="correct")
A, B = "Laya (fine-tuned, 3k)", "Phi-4-mini (zero-shot)"
only_a, only_b = int((wide[A] & ~wide[B]).sum()), int((~wide[A] & wide[B]).sum())
p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
acc_a, acc_b = wide[A].mean(), wide[B].mean()
print(f"{A}: {acc_a:.1%} | {B}: {acc_b:.1%} | difference {100 * (acc_a - acc_b):+.1f} pts")
print(f"Only Laya right: {only_a} | only Phi right: {only_b} | exact McNemar p = {p:.2g}")
if p >= 0.05:
    print("Verdict: no significant difference at the 0.05 level.")
else:
    print(f"Verdict: {'fine-tuned Laya' if acc_a > acc_b else 'Phi-4-mini'} is significantly better (p < 0.05).")
base = wide["Laya (zero-shot)"].mean()
print(f"Fine-tuning moved Laya {100 * (acc_a - base):+.1f} pts ({base:.1%} -> {acc_a:.1%}).")

## 9. Figures

In [ ]:
COLORS = {"Laya (zero-shot)": "#9ecae9", "Laya (fine-tuned, 3k)": "#4C78A8", "Phi-4-mini (zero-shot)": "#F58518"}
names = list(COLORS)
s = summary.loc[names]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

ax = axes[0]
ax.bar(range(3), s.accuracy, yerr=[s.accuracy - s.ci_low, s.ci_high - s.accuracy], capsize=6,
       color=[COLORS[m] for m in names])
ax.axhline(max(MAJORITY, 1 - MAJORITY), color="grey", ls="--", label="majority class")
ax.set_xticks(range(3), [m.replace(" (", "\n(") for m in names])
ax.set_ylim(0.5, 1); ax.set_ylabel("Accuracy"); ax.legend(loc="upper left")
ax.set_title(f"BoolQ validation (n={len(test_df)}, 95% Wilson CI)")
for i, v in enumerate(s.accuracy):
    ax.text(i, v + 0.012, f"{v:.1%}", ha="center")

ax = axes[1]
h = pd.DataFrame(history)
if len(h):
    ax.plot(h.epoch, h.calib_acc, "o-", color=COLORS["Laya (fine-tuned, 3k)"], label="Laya held-out calib accuracy")
    ax.axhline(summary.loc["Phi-4-mini (zero-shot)", "accuracy"], color=COLORS["Phi-4-mini (zero-shot)"], ls="--",
               label="Phi-4-mini (validation)")
    ax.axhline(summary.loc["Laya (zero-shot)", "accuracy"], color=COLORS["Laya (zero-shot)"], ls=":",
               label="Laya zero-shot (validation)")
    ax.set_xticks(h.epoch); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
ax.set_title("Fine-tuning progress")

ax = axes[2]
edges = np.linspace(0, 1, 11)
for m in names:
    g = res[res.model == m]
    b = np.clip(np.digitize(g.p_yes, edges[1:-1]), 0, 9)
    cal = pd.DataFrame({"b": b, "p": g.p_yes, "y": g.label.astype(float)}).groupby("b").mean()
    ax.plot(cal.p, cal.y, "o-", color=COLORS[m], label=f"{m} (ECE {summary.loc[m, 'ece']:.3f})")
ax.plot([0, 1], [0, 1], color="grey", ls=":")
ax.set_xlabel("Predicted P(yes)"); ax.set_ylabel("Observed Yes rate"); ax.legend(fontsize=8)
ax.set_title("Reliability diagram")

plt.tight_layout()
fig.savefig(FIG_DIR / "finetune_vs_phi4.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Save results

Download the zip and commit `figures/`, `results.csv` and `summary.csv` to this experiment folder. The fine-tuned checkpoint in `laya_boolq_ft/` (~850 MB) is not included. Set `USE_DRIVE = True` to keep it.

In [ ]:
import shutil
bundle = Path("/tmp/laya_finetune_boolq_results")
if bundle.exists():
    shutil.rmtree(bundle)
for name in ["cache", "figures"]:
    shutil.copytree(BASE_DIR / name, bundle / name)
for name in ["results.csv", "summary.csv"]:
    shutil.copy(BASE_DIR / name, bundle / name)
shutil.make_archive("laya_finetune_boolq_results", "zip", bundle)
try:
    from google.colab import files
    files.download("laya_finetune_boolq_results.zip")
except ImportError:
    print("Not on Colab; results in", BASE_DIR.resolve())